In [3]:
# packages and working directory  
import scipy 
import sklearn
import econml 
import arch
import os 
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib as plt
import statsmodels.api as sm 
from statsmodels.discrete.discrete_model import Probit
from statsmodels.iolib.summary2 import summary_col
import statsmodels.formula.api as smf 
from scipy.optimize import minimize
from scipy.special import logsumexp
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt 
from scipy import stats
from scipy.stats import ttest_ind
from scipy.optimize import approx_fprime
# Consolidate changing directory and CPI dictionary since these don't change throughout the script
new_directory = r'C:\Users\hisham\Spain\2021 datasets'
os.chdir(new_directory)


In [2]:
df = pd.read_csv('manychoicestalong.csv')

In [3]:
# Define the list of scenarios
scenarios = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

# Create the 'labor' column based on the 'scenario' column
df['labor'] = df.apply(lambda row: row[f'lhw_{int(row["scenario"])}'], axis=1)


In [15]:
df['log_y'] = np.log(df['ils_udb_yds'])
df['log_l'] = np.log(80 - df['labor'])
df['log2_y'] =  0.5* df['log_y']**2 
df['log2_l'] =  0.5 *df['log_l']**2
df['log_y_l'] = df['log_y'] * df['log_l']


In [9]:

df[['idperson', 'labor','lhw','ils_udb_yds','log_y','log_l','log2_y','log2_l','log_y_l','scenario','choice_made','original_scenario']].describe()

,idperson,labor,lhw,ils_udb_yds,log_y,log_l,log2_y,log2_l,log_y_l,scenario,choice_made,original_scenario
count,6.870400e+04,68704.000000,68704.000000,68704.000000,68704.000000,68704.000000,68704.000000,68704.000000,68704.000000,68704.000000,68704.000000,68704.000000
mean,4.218787e+08,36.113793,36.537727,1853.031136,7.278179,3.584039,26.739857,6.672483,25.773490,7.500000,0.062500,7.402655
std,1.559576e+08,22.912314,12.509830,1395.697055,0.712627,0.706849,5.186260,2.308488,4.283821,4.609806,0.242063,2.505011
min,8.701000e+07,0.000000,0.000000,250.250000,5.522460,1.609438,15.248785,1.295145,10.250885,0.000000,0.000000,0.000000
25%,2.773500e+08,15.750000,35.000000,830.400000,6.721908,3.208670,22.592020,5.147939,24.124699,3.750000,0.000000,7.000000
50%,4.812400e+08,35.500000,40.000000,1523.430000,7.328720,3.795426,26.855066,7.202693,26.724805,7.500000,0.000000,8.000000
75%,5.550400e+08,55.250000,40.000000,2459.437500,7.807688,4.162759,30.479996,8.664304,28.463421,11.250000,0.000000,8.000000
max,6.201200e+08,75.000000,75.000000,24748.610000,10.116525,4.382027,51.172035,9.601079,39.544696,15.000000,1.000000,15.000000


In [10]:
# Assuming 'df' is your DataFrame
df.to_stata('manychoices.dta')


In [16]:
df['log_l'].mean()

3.58403944915223

In [21]:
monet_vars = df[['dag',  'dwt', 'yivwg', 'ils_b1_bsa',  'il_bsa00',   'ils_udb_bun',  'ils_udb_bhl',  'ils_b1_bcb', 'kfb',
              'ils_udb_yiy',  'ils_udb_ypp',  'ils_udb_ypr',  'ils_udb_ypt', 'il_bsa00', 'il_bsarg_global',  'il_bsarg_11', 
              'il_bsarg_12', 'il_bsarg_13', 'il_bsarg_21', 'il_bsarg_22', 'il_bsarg_23','il_bsarg_24', 'il_bsarg_30', 'il_bsarg_41', 'il_bsarg_42',  'il_bsarg_43',
               'il_bsarg_51', 'il_bsarg_52',  'il_bsarg_53',  'il_bsarg_61',  'il_bsarg_62',  'il_bsarg_63',  'il_bsarg_64',  'il_bsarg_70',  'yptmp',  'afc', 'ils_udb_xmp',
              'xpp', 'xhcmomi', 'xhcmomc', 'xhcmo', 'xhcrt','xhc','xed00','xhl00', 'xhcot']]

In [23]:
print(monet_vars.columns)

Index(['dag', 'dwt', 'yivwg', 'ils_b1_bsa', 'il_bsa00', 'ils_udb_bun',
       'ils_udb_bhl', 'ils_b1_bcb', 'kfb', 'ils_udb_yiy', 'ils_udb_ypp',
       'ils_udb_ypr', 'ils_udb_ypt', 'il_bsa00', 'il_bsarg_global',
       'il_bsarg_11', 'il_bsarg_12', 'il_bsarg_13', 'il_bsarg_21',
       'il_bsarg_22', 'il_bsarg_23', 'il_bsarg_24', 'il_bsarg_30',
       'il_bsarg_41', 'il_bsarg_42', 'il_bsarg_43', 'il_bsarg_51',
       'il_bsarg_52', 'il_bsarg_53', 'il_bsarg_61', 'il_bsarg_62',
       'il_bsarg_63', 'il_bsarg_64', 'il_bsarg_70', 'yptmp', 'afc',
       'ils_udb_xmp', 'xpp', 'xhcmomi', 'xhcmomc', 'xhcmo', 'xhcrt', 'xhc',
       'xed00', 'xhl00', 'xhcot'],
      dtype='object')


In [25]:
categorical_vars =  df[['dcz','dgn',  'lcs', 'drgru', 'drgur','dms', 'deh', 'lindi', 'loc', 'aca', 'aco', 'amrrm', 'amrtn']]

In [26]:
categorical_vars.describe()

,dcz,dgn,lcs,drgru,drgur,dms,deh,lindi,loc,aca,aco,amrrm,amrtn
count,68704.000000,68704.000000,68704.000000,68704.000000,68704.000000,68704.000000,68704.000000,68704.000000,68704.000000,68704.000000,68704.000000,68704.000000,68704.000000
mean,1.090592,0.442012,0.106428,0.147415,0.562180,2.094551,3.494877,6.383093,4.404285,1.336283,1.436889,4.648114,2.175361
std,0.385994,0.496630,0.308386,0.354522,0.496122,1.367645,1.520358,3.689689,2.649999,0.711284,0.786536,1.429074,1.258881
min,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,-1.000000,1.000000,1.000000,1.000000,1.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2.000000,3.000000,2.000000,1.000000,1.000000,4.000000,1.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,3.000000,6.000000,4.000000,1.000000,1.000000,5.000000,2.000000
75%,1.000000,1.000000,0.000000,0.000000,1.000000,4.000000,5.000000,10.000000,7.000000,1.000000,2.000000,5.000000,3.000000
max,3.000000,1.000000,1.000000,1.000000,1.000000,5.000000,5.000000,12.000000,9.000000,3.000000,3.000000,8.000000,6.000000


In [28]:
categorical_vars[categorical_vars['loc'] == -1 ]

,dcz,dgn,lcs,drgru,drgur,dms,deh,lindi,loc,aca,aco,amrrm,amrtn
2,1,0,0,1,0,5,2,0,-1,3,3,4,2
20,1,0,0,0,1,5,0,0,-1,3,3,5,2
37,1,0,0,0,1,5,0,0,-1,3,3,3,2
38,1,0,0,0,1,5,0,0,-1,1,3,5,6
51,1,0,0,0,1,5,5,0,-1,2,3,3,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
68582,1,0,0,0,0,5,0,0,-1,3,3,2,3
68641,1,1,0,0,1,2,0,0,-1,3,2,5,2
68642,1,0,0,0,1,5,1,0,-1,3,3,6,2
68680,1,0,0,0,0,1,0,0,-1,3,1,4,3


In [12]:
def utility(params, row):
    # Unpack parameters
    alpha, beta, gamma_yy, gamma_ll, gamma_yl = params
    # Use precomputed log values directly from the row
    log_y = row['log_y']
    log_l = row['log_l']
    log2_y = row['log2_y']
    log2_l = row['log2_l']
    log_y_l = row['log_y_l']
    # Translog utility function calculation
    return alpha * log_y + beta * log_l + gamma_yy * log2_y + gamma_ll * log2_l + gamma_yl * log_y_l

def ind_likelihood(params, group):
    # Calculate utility for all scenarios in the group
    utilities = group.apply(lambda row: utility(params, row), axis=1)
    # Select the utility of the chosen scenario
    chosen_utility = utilities[group['choice_made'] == 1].iloc[0] if any(group['choice_made'] == 1) else float('-inf')
    # Compute the log likelihood of the chosen utility
    if chosen_utility > float('-inf'):  # Ensures there is a chosen utility
        log_prob_chosen = chosen_utility - logsumexp(utilities)
        return -log_prob_chosen  # Return negative log likelihood
    else:
        return 0  # Return 0 likelihood if no choice is made

def total_likelihood(params, df):
    # Group by 'idperson' and compute the likelihood for each group
    return df.groupby('idperson').apply(lambda group: ind_likelihood(params, group)).sum()

In [13]:

initial_params = [20.55,  -2.46196, 72.5843, -34.54132,  -.8358232] 

# Optimization
result1 = minimize(fun=total_likelihood, x0=initial_params, args=(df,), method='BFGS')
print(result1)

  message: Desired error not necessarily achieved due to precision loss.
  success: False
   status: 2
      fun: 9797.14503469947
        x: [ 8.953e+00  3.836e+01 -8.974e-01 -9.543e+00 -3.631e-01]
      nit: 28
      jac: [ 9.766e-04 -1.221e-03  6.226e-03 -4.883e-03 -8.911e-03]
 hess_inv: [[ 7.060e+00  3.316e+00 ... -6.911e-01 -1.144e-01]
            [ 3.316e+00  1.876e+00 ... -4.204e-01 -5.143e-02]
            ...
            [-6.911e-01 -4.204e-01 ...  1.065e-01  6.338e-03]
            [-1.144e-01 -5.143e-02 ...  6.338e-03  3.802e-03]]
     nfev: 545
     njev: 89


In [6]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp

# Consolidate changing directory and CPI dictionary since these don't change throughout the script
new_directory = r'C:\Users\hisham\Spain\2021 datasets'
os.chdir(new_directory)

# Load data
df = pd.read_csv('manychoicestalong.csv')




# Define the list of scenarios
scenarios = list(range(16))  # Adjust based on actual scenarios



In [7]:
# Precompute necessary log values and interactions directly
df['log_y'] = np.log(df['ils_udb_yds'])
df['log_l'] = np.log(80 - df['lhw'])  # Adjust based on your 'lhw' column
df['log2_y'] = df['log_y'] ** 2
df['log2_l'] = df['log_l'] ** 2
df['log_y_l'] = df['log_y'] * df['log_l']

# Utility function using direct DataFrame operations
def utility(params, row):
    alpha, beta, gamma_yy, gamma_ll, gamma_yl = params
    return (alpha * row['log_y'] + beta * row['log_l'] +
            gamma_yy * row['log2_y'] + gamma_ll * row['log2_l'] + gamma_yl * row['log_y_l'])

# Vectorized likelihood calculation
def ind_likelihood(params, group):
    utilities = group.apply(lambda row: utility(params, row), axis=1)
    chosen_utility = utilities[group['choice_made'] == 1].max()
    log_sum_exp = logsumexp(utilities)
    return -(chosen_utility - log_sum_exp)

# Total likelihood for the dataset
def total_likelihood(params, df):
    grouped = df.groupby('idperson')
    total_ll = sum(grouped.apply(lambda g: ind_likelihood(params, g)))
    return total_ll


In [8]:
# Initial parameter guesses
initial_params = [ 8.953e+00,  3.836e+01, -4.974e-01, -4.543e+00, -3.631e-01]
# Optimization using 'L-BFGS-B' because it handles large scale problems well
result = minimize(fun=total_likelihood, x0=initial_params, args=(df,), method='L-BFGS-B')
print(result)



  message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
  success: True
   status: 0
      fun: 10539.349718424814
        x: [ 5.705e+01  3.837e+01 -1.755e+00 -4.545e+00 -8.239e+00]
      nit: 24
      jac: [ 1.291e-02  2.001e-03  1.044e-01  0.000e+00  2.910e-02]
     nfev: 180
     njev: 30
 hess_inv: <5x5 LbfgsInvHessProduct with dtype=float64>


In [15]:
print(df['log_l'].mean())
print(df['log_y'].mean())

3.7317501085043765
7.278178616091539


In [10]:
# Automated calculation of points where marginal utilities become negative
mu_y_neg = np.exp(- (result.x[0] + result.x[4] * df['log_l'].mean()) / (2 * result.x[2]))
mu_l_neg = np.exp(- (result.x[1] + result.x[4] * df['log_y'].mean()) / (2 * result.x[3]))

print(f"Income level at which MU of income becomes negative: {mu_y_neg}")
print(f"Leisure level at which MU of leisure becomes negative: {mu_l_neg}")

Income level at which MU of income becomes negative: 1792.325563526017
Leisure level at which MU of leisure becomes negative: 0.09285485410186407


In [28]:
# Automated calculation of points where marginal utilities become negative
mu_y_neg = np.exp(- (result1.x[0] + result.x[4] * df['log_l'].mean()) / (2 * result1.x[2]))
mu_l_neg = np.exp(- (result1.x[1] + result1.x[4] * df['log_y'].mean()) / (2 * result1.x[3]))

print(f"Income level at which MU of income becomes negative: {mu_y_neg}")
print(f"Leisure level at which MU of leisure becomes negative: {mu_l_neg}")

NameError: name 'result1' is not defined

In [20]:
def utility(params, row):
    # Unpack parameters
    alpha, beta, gamma_yy, gamma_ll, gamma_yl = params
    # Use precomputed log values directly from the row
    log_y = row['log_y']
    log_l = row['log_l']
    log2_y = row['log2_y']
    log2_l = row['log2_l']
    log_y_l = row['log_y_l']
    # Translog utility function calculation
    return alpha * log_y + beta * log_l + gamma_yy * log2_y + gamma_ll * log2_l + gamma_yl * log_y_l

def ind_likelihood(params, group):
    # Calculate utility for all scenarios in the group
    utilities = group.apply(lambda row: utility(params, row), axis=1)
    # Select the utility of the chosen scenario
    chosen_utility = utilities[group['choice_made'] == 1].iloc[0] if any(group['choice_made'] == 1) else float('-inf')
    # Compute the log likelihood of the chosen utility
    if chosen_utility > float('-inf'):  # Ensures there is a chosen utility
        log_prob_chosen = chosen_utility - logsumexp(utilities)
        return -log_prob_chosen  # Return negative log likelihood
    else:
        return 0  # Return 0 likelihood if no choice is made

def total_likelihood(params, df):
    # Group by 'idperson' and compute the likelihood for each group
    return df.groupby('idperson').apply(lambda group: ind_likelihood(params, group)).sum()

In [21]:
initial_params = [ 8.953e+00,  3.836e+01, -8.974e-01, -9.543e+00, -3.631e-01]  # Example: alpha, beta, gamma_yy, gamma_ll, gamma_yl

    # Optimization
result = minimize(fun=total_likelihood, x0=initial_params, args=(df,), method='BFGS')
print(result)


  message: Desired error not necessarily achieved due to precision loss.
  success: False
   status: 2
      fun: 10539.349706697656
        x: [ 5.704e+01  3.836e+01 -1.755e+00 -9.543e+00 -8.238e+00]
      nit: 19
      jac: [ 3.662e-04  0.000e+00  1.343e-03 -3.662e-04  2.441e-04]
 hess_inv: [[ 7.339e-02  1.400e-02 ...  9.842e-02 -3.808e-01]
            [ 1.400e-02  2.695e-03 ...  1.876e-02 -7.265e-02]
            ...
            [ 9.842e-02  1.876e-02 ...  1.320e-01 -5.111e-01]
            [-3.808e-01 -7.265e-02 ... -5.111e-01  1.979e+00]]
     nfev: 222
     njev: 37


In [23]:
# Automated calculation of points where marginal utilities become negative
mu_y_neg = np.exp(- (result.x[0] + result.x[4] * df['log_l'].mean()) / (2 * result.x[2]))
mu_l_neg = np.exp(- (result.x[1] + result.x[4] * df['log_y'].mean()) / (2 * result.x[3]))

print(f"Income level at which MU of income becomes negative: {mu_y_neg}")
print(f"Leisure level at which MU of leisure becomes negative: {mu_l_neg}")

Income level at which MU of income becomes negative: 1792.334733714019
Leisure level at which MU of leisure becomes negative: 0.32249485273914374


In [24]:
result.x[0]

57.038034494092706

In [27]:
res1 =[ 8.953e+00,  3.836e+01, -8.974e-01, -9.543e+00, -3.631e-01]
# Automated calculation of points where marginal utilities become negative
mu_y_neg = np.exp(- (res1[0] + res1[4] * df['log_l'].mean()) / ( res1[2]))
mu_l_neg = np.exp(- (res1[1] + res1[4] * df['log_y'].mean()) / ( res1[3]))

print(f"Income level at which MU of income becomes negative: {mu_y_neg}")
print(f"Leisure level at which MU of leisure becomes negative: {mu_l_neg}")

Income level at which MU of income becomes negative: 4753.722315805779
Leisure level at which MU of leisure becomes negative: 42.21493681332735


In [31]:
# Define the list of scenarios
scenarios = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

# Create the 'labor' column based on the 'scenario' column
df['labor'] = df.apply(lambda row: row[f'lhw_{int(row["scenario"])}'], axis=1)


In [37]:
df['y'] = df['ils_udb_yds']
df['l'] = 80 - df['labor']
df['y2'] = df['ils_udb_yds'] ** 2 
df['l2'] = 80 - df['labor'] ** 2
df['y_l'] = df['l']  *  df['y']

In [43]:
def utility(params, row):
    # Unpack parameters
    alpha, beta, gamma_yy, gamma_ll, gamma_yl = params
    # Use precomputed log values directly from the row
    y = row['y']
    l = row['l']
    y2 = row['y2']
    l2 = row['l2']
    y_l = row['y_l']
    # Translog utility function calculation
    return alpha * y + beta * l + gamma_yy * y2 + gamma_ll * l2 + gamma_yl * y_l

def ind_likelihood(params, group):
    # Calculate utility for all scenarios in the group
    utilities = group.apply(lambda row: utility(params, row), axis=1)
    # Select the utility of the chosen scenario
    chosen_utility = utilities[group['choice_made'] == 1].iloc[0] if any(group['choice_made'] == 1) else float('-inf')
    # Compute the log likelihood of the chosen utility
    if chosen_utility > float('-inf'):  # Ensures there is a chosen utility
        log_prob_chosen = chosen_utility - logsumexp(utilities)
        return -log_prob_chosen  # Return negative log likelihood
    else:
        return 0  # Return 0 likelihood if no choice is made

def total_likelihood(params, df):
    # Group by 'idperson' and compute the likelihood for each group
    return df.groupby('idperson').apply(lambda group: ind_likelihood(params, group)).sum()

In [44]:

initial_params = [.0008725,  .2841522 , -9.78e-08, -.0030662 , 3.33e-08] 

# Optimization
result = minimize(fun=total_likelihood, x0=initial_params, args=(df,), method='BFGS')
print(result)

  message: Desired error not necessarily achieved due to precision loss.
  success: False
   status: 2
      fun: 9999.185271359216
        x: [ 1.588e-03 -1.999e-01 -1.772e-07  3.134e-03 -5.004e-06]
      nit: 12
      jac: [ 8.501e+03 -2.440e+02  8.006e+07 -1.836e+04 -2.588e+05]
 hess_inv: [[ 8.132e-09  5.660e-08 ...  2.337e-09 -9.876e-11]
            [ 5.660e-08  1.382e-05 ... -1.585e-07  1.984e-09]
            ...
            [ 2.337e-09 -1.585e-07 ...  3.299e-09 -7.131e-11]
            [-9.876e-11  1.984e-09 ... -7.131e-11  2.958e-12]]
     nfev: 390
     njev: 63
